In [1]:
from src.utils.paths import load_paths
import pandas as pd

paths = load_paths()
preds_path = paths.artifacts_xgb / "preds.parquet"

if not preds_path.exists():
    print(f"File not found: {preds_path}")
    print("Please run notebook 04_train_xgboost.ipynb first to generate predictions.")
else:
    df = pd.read_parquet(preds_path)
    print(df.head())
    print(df.columns)

   split  label                          flow_id  \
0  train      1     vpn_youtube_capture2.pcap::0   
1  train      0  nonvpn_sftp_newcapture1.pcap::1   
2  train      0  nonvpn_sftp_newcapture1.pcap::2   
3  train      0  nonvpn_sftp_newcapture1.pcap::3   
4  train      0  nonvpn_sftp_newcapture1.pcap::4   

                     capture_id     p_xgb     p_raw  
0     vpn_youtube_capture2.pcap  0.999519  0.999519  
1  nonvpn_sftp_newcapture1.pcap  0.000045  0.000045  
2  nonvpn_sftp_newcapture1.pcap  0.000049  0.000049  
3  nonvpn_sftp_newcapture1.pcap  0.000571  0.000571  
4  nonvpn_sftp_newcapture1.pcap  0.000048  0.000048  
Index(['split', 'label', 'flow_id', 'capture_id', 'p_xgb', 'p_raw'], dtype='str')


In [2]:
from src.utils.paths import load_paths
from src.utils.logging import setup_logger

import pandas as pd
import numpy as np
import json
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix

paths = load_paths()
logger = setup_logger(level="INFO")

preds_path = paths.artifacts_xgb / "preds.parquet"
metrics_path = paths.artifacts_xgb / "metrics_calibrated.json"
calib_model_path = paths.artifacts_xgb / "calibrator.pkl"

df = pd.read_parquet(preds_path)

print("Loaded:", preds_path)
print(df["split"].value_counts())

Loaded: C:\Users\scoti\PycharmProjects\ai-vpn-firewall\artifacts\xgb\preds.parquet
split
train    32914
test       464
val        333
Name: count, dtype: int64


In [3]:
train_df = df[df["split"] == "train"].copy()
val_df   = df[df["split"] == "val"].copy()
test_df  = df[df["split"] == "test"].copy()

print("VAL positives:", val_df["label"].sum())

VAL positives: 12


In [4]:
X_val = val_df["p_xgb"].values.reshape(-1, 1)
y_val = val_df["label"].values

calibrator = LogisticRegression(solver="lbfgs")
calibrator.fit(X_val, y_val)

print("Calibration coefficients:", calibrator.coef_, calibrator.intercept_)

Calibration coefficients: [[4.4731457]] [-4.26460083]


In [5]:
def apply_calibration(df_split):
    X = df_split["p_xgb"].values.reshape(-1, 1)
    df_split["p_calib"] = calibrator.predict_proba(X)[:, 1]
    return df_split

train_df = apply_calibration(train_df)
val_df   = apply_calibration(val_df)
test_df  = apply_calibration(test_df)

In [6]:
def evaluate(split_df, prob_col):
    y = split_df["label"].values
    p = split_df[prob_col].values

    roc = roc_auc_score(y, p)
    pr  = average_precision_score(y, p)

    y_hat = (p >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, y_hat).ravel()

    return {
        "roc_auc": float(roc),
        "pr_auc": float(pr),
        "threshold_0.5": {
            "precision": float(tp / (tp + fp + 1e-9)),
            "recall": float(tp / (tp + fn + 1e-9)),
            "confusion_matrix": {
                "tn": int(tn),
                "fp": int(fp),
                "fn": int(fn),
                "tp": int(tp)
            }
        }
    }

In [7]:
results = {
    "raw": {
        "val": evaluate(val_df, "p_xgb"),
        "test": evaluate(test_df, "p_xgb"),
    },
    "calibrated": {
        "val": evaluate(val_df, "p_calib"),
        "test": evaluate(test_df, "p_calib"),
    }
}

print(json.dumps(results, indent=2))

{
  "raw": {
    "val": {
      "roc_auc": 0.9997403946002077,
      "pr_auc": 0.9935897435897437,
      "threshold_0.5": {
        "precision": 0.7999999999466667,
        "recall": 0.9999999999166667,
        "confusion_matrix": {
          "tn": 318,
          "fp": 3,
          "fn": 0,
          "tp": 12
        }
      }
    },
    "test": {
      "roc_auc": 1.0,
      "pr_auc": 0.9999999999999998,
      "threshold_0.5": {
        "precision": 0.8124999999492187,
        "recall": 0.9999999999230769,
        "confusion_matrix": {
          "tn": 448,
          "fp": 3,
          "fn": 0,
          "tp": 13
        }
      }
    }
  },
  "calibrated": {
    "val": {
      "roc_auc": 0.9997403946002077,
      "pr_auc": 0.9935897435897437,
      "threshold_0.5": {
        "precision": 0.9166666665902777,
        "recall": 0.9166666665902777,
        "confusion_matrix": {
          "tn": 320,
          "fp": 1,
          "fn": 1,
          "tp": 11
        }
      }
    },
    "test"

In [8]:
import joblib

joblib.dump(calibrator, calib_model_path)

with open(metrics_path, "w") as f:
    json.dump(results, f, indent=2)

print("Saved calibrator to:", calib_model_path)
print("Saved metrics to:", metrics_path)

Saved calibrator to: C:\Users\scoti\PycharmProjects\ai-vpn-firewall\artifacts\xgb\calibrator.pkl
Saved metrics to: C:\Users\scoti\PycharmProjects\ai-vpn-firewall\artifacts\xgb\metrics_calibrated.json
